In [6]:
import pandas as pd
import requests
from datetime import datetime
from pathlib import Path

pd.set_option("display.max_columns", None)


In [7]:
# -- 1. PATH CONFIGURATION ---------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = BASE_DATA_DIR / "raw"
BRONZE_DIR = BASE_DATA_DIR / "bronze"
SILVER_DIR = BASE_DATA_DIR / "silver"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://www.data.gouv.fr/api/1/datasets/r/f81b2215-b297-4616-acbf-d8790ee38197"

path_xls_raw = RAW_DIR / "2012_pres_t1_t2_communes_france.xls"
path_t1_rhone_bronze = BRONZE_DIR / "2012-pres-t1-commune-rhone-69-bronze.csv"
path_t2_rhone_bronze = BRONZE_DIR / "2012-pres-t2-commune-rhone-69-bronze.csv"
path_t1_rhone_silver = SILVER_DIR / "2012-pres-t1-commune-rhone-69-silver.csv"
path_t2_rhone_silver = SILVER_DIR / "2012-pres-t2-commune-rhone-69-silver.csv"

print(f"Raw data path: {path_xls_raw}")
print(f"Tour 1 bronze path: {path_t1_rhone_bronze}")
print(f"Tour 2 bronze path: {path_t2_rhone_bronze}")
print(f"Tour 1 silver path: {path_t1_rhone_silver}")
print(f"Tour 2 silver path: {path_t2_rhone_silver}")


Raw data path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/raw/2012_pres_t1_t2_communes_france.xls
Tour 1 bronze path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/bronze/2012-pres-t1-commune-rhone-69-bronze.csv
Tour 2 bronze path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/bronze/2012-pres-t2-commune-rhone-69-bronze.csv
Tour 1 silver path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/2012-pres-t1-commune-rhone-69-silver.csv
Tour 2 silver path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/2012-pres-t2-commune-rhone-69-silver.csv


In [8]:
# -- 2. RAW LAYER: Landing Zone ----------------------------------------------
if not path_xls_raw.exists():
    print("Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_xls_raw, "wb") as f:
        f.write(resp.content)
    print(f"Saved to RAW: {path_xls_raw}")
else:
    print(f"Raw file already exists in {RAW_DIR}")


Raw file already exists in /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/raw


In [9]:
# -- 3. BRONZE LAYER: Ingestion & Metadata -----------------------------------
print("\nProcessing RAW -> BRONZE...")

CANDIDATE_BASE_FIELDS = ["Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]


def normalize_candidate_columns(df: pd.DataFrame) -> pd.DataFrame:
    renamed_columns = []

    for col in df.columns:
        if col in CANDIDATE_BASE_FIELDS:
            renamed_columns.append(f"{col}_1")
            continue

        matched = False
        for field in CANDIDATE_BASE_FIELDS:
            prefix = f"{field}."
            if col.startswith(prefix):
                suffix = col[len(prefix):]
                if suffix.isdigit():
                    renamed_columns.append(f"{field}_{int(suffix) + 1}")
                    matched = True
                    break

        if not matched:
            renamed_columns.append(col)

    df = df.copy()
    df.columns = renamed_columns
    return df


def build_bronze_from_sheet(sheet_name: str) -> pd.DataFrame:
    df_all = pd.read_excel(path_xls_raw, sheet_name=sheet_name, header=0, dtype=str, engine="xlrd")
    df_all = df_all.dropna(axis=0, how="all").dropna(axis=1, how="all")

    if df_all.empty:
        raise ValueError(f"Sheet '{sheet_name}' is empty.")

    col_dept = next(
        (
            c
            for c in df_all.columns
            if any(kw in str(c).lower() for kw in ["département", "departement", "dept"])
        ),
        None,
    )

    if col_dept is None:
        raise ValueError(f"Department column not found in sheet '{sheet_name}'.")

    df_bronze = df_all[
        df_all[col_dept].astype(str).str.strip().str.lstrip("0") == "69"
    ].copy()
    df_bronze = normalize_candidate_columns(df_bronze)
    df_bronze["source_sheet_name"] = sheet_name
    df_bronze["extraction_source_url"] = DATA_URL
    df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
    df_bronze["source_file_name"] = path_xls_raw.name
    return df_bronze


xls = pd.ExcelFile(path_xls_raw, engine="xlrd")
sheet_names = []

for sheet_name in xls.sheet_names:
    preview = pd.read_excel(path_xls_raw, sheet_name=sheet_name, header=0, dtype=str, engine="xlrd", nrows=5)
    preview = preview.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if not preview.empty:
        sheet_names.append(sheet_name)

sheet_names = sheet_names[:2]

if len(sheet_names) != 2:
    raise ValueError(f"Expected 2 non-empty sheets, found {len(sheet_names)}: {sheet_names}")

df_t1_bronze = build_bronze_from_sheet(sheet_names[0])
df_t2_bronze = build_bronze_from_sheet(sheet_names[1])

print(f"Detected sheets: {sheet_names}")
print(f"Tour 1 columns: {df_t1_bronze.columns.to_list()}")
print(f"Tour 2 columns: {df_t2_bronze.columns.to_list()}")



Processing RAW -> BRONZE...
Detected sheets: ['Tour 1', 'Tour 2']
Tour 1 columns: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs et nuls', '% BlNuls/Ins', '% BlNuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'Sexe_1', 'Nom_1', 'Prénom_1', 'Voix_1', '% Voix/Ins_1', '% Voix/Exp_1', 'Sexe_2', 'Nom_2', 'Prénom_2', 'Voix_2', '% Voix/Ins_2', '% Voix/Exp_2', 'Sexe_3', 'Nom_3', 'Prénom_3', 'Voix_3', '% Voix/Ins_3', '% Voix/Exp_3', 'Sexe_4', 'Nom_4', 'Prénom_4', 'Voix_4', '% Voix/Ins_4', '% Voix/Exp_4', 'Sexe_5', 'Nom_5', 'Prénom_5', 'Voix_5', '% Voix/Ins_5', '% Voix/Exp_5', 'Sexe_6', 'Nom_6', 'Prénom_6', 'Voix_6', '% Voix/Ins_6', '% Voix/Exp_6', 'Sexe_7', 'Nom_7', 'Prénom_7', 'Voix_7', '% Voix/Ins_7', '% Voix/Exp_7', 'Sexe_8', 'Nom_8', 'Prénom_8', 'Voix_8', '% Voix/Ins_8', '% Voix/Exp_8', 'Sexe_9', 'Nom_9', 'Prénom_9', 'Voix_9', '% Voix/Ins_9', '% Voix/Exp_9', 'Sexe_10', 

In [10]:
# -- 4. SAVE TO BRONZE -------------------------------------------------------
df_t1_bronze.to_csv(path_t1_rhone_bronze, index=False, sep=";", encoding="utf-8")
df_t2_bronze.to_csv(path_t2_rhone_bronze, index=False, sep=";", encoding="utf-8")

print(df_t1_bronze.head())
print(df_t2_bronze.head())

print(f"BRONZE dataset created: {path_t1_rhone_bronze}")
print(f"Rows: {len(df_t1_bronze)} | Columns: {len(df_t1_bronze.columns)}")
print(f"BRONZE dataset created: {path_t2_rhone_bronze}")
print(f"Rows: {len(df_t2_bronze)} | Columns: {len(df_t2_bronze.columns)}")


      Code du département Libellé du département Code de la commune  \
28410                  69                  RHONE                  1   
28411                  69                  RHONE                  2   
28412                  69                  RHONE                  3   
28413                  69                  RHONE                  4   
28414                  69                  RHONE                  5   

      Libellé de la commune Inscrits Abstentions % Abs/Ins Votants % Vot/Ins  \
28410                Affoux      250          25        10     225        90   
28411            Aigueperse      224          42     18.75     182     81.25   
28412     Albigny-sur-Saône     1583         272     17.18    1311     82.82   
28413                  Alix      417          49     11.75     368     88.25   
28414             Ambérieux      389          49      12.6     340      87.4   

      Blancs et nuls % BlNuls/Ins % BlNuls/Vot Exprimés % Exp/Ins % Exp/Vot  \
28410        